## Getting Started

### Install Google Gen AI SDK for Python


In [1]:
!python -m pip install --upgrade --quiet google-genai

    gcloud init

step 1 - select mode : 2 (sign in with new google account)
- it would launch a browser for you to eneter your google cloud credentials, 
- allow all, next, next, complete
- step 2: select project: project 1 - bdc-training
- step 3: region for VM: uscentral1-b


        gcloud auth application-default login

In [1]:
import os
os.getenv('GOOGLE_APPLICATION_CREDENTIALS')

### Connect to a generative AI API service

Google Gen AI APIs and models including Gemini are available in the following two API services:

- **[Google AI for Developers](https://ai.google.dev/gemini-api/docs)**: Experiment, prototype, and deploy small projects.
- **[Vertex AI](https://cloud.google.com/vertex-ai/generative-ai/docs/overview)**: Build enterprise-ready projects on Google Cloud.

The Google Gen AI SDK provides a unified interface to these two API services.

This notebook shows how to use the Google Gen AI SDK with the Gemini API in Vertex AI.

### Import libraries


In [2]:
from IPython.display import HTML, Markdown, display
from google import genai
from google.genai.types import (
    FunctionDeclaration,
    GenerateContentConfig,
    GoogleSearch,
    HarmBlockThreshold,
    HarmCategory,
    MediaResolution,
    Part,
    SafetySetting,
    Tool,
    ToolCodeExecution,
)

### Set up Google Cloud Project or API Key for Vertex AI

You'll need to set up authentication by choosing **one** of the following methods:

1.  **Use a Google Cloud Project:** Recommended for most users, this requires enabling the Vertex AI API in your Google Cloud project.
    - [Enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com)
    - Run the cell below to set your project ID and location.
    - Read more about [Supported locations](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/locations)
2.  **Use a Vertex AI API Key (Express Mode):** For quick experimentation.
    - [Get an API Key](https://cloud.google.com/vertex-ai/generative-ai/docs/start/express-mode/overview)
    - Run the cell further below to use your API key.

#### Option 1. Use a Google Cloud Project


In [3]:
import os

PROJECT_ID = "bdc-trainings"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "global")

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

#### Option 2. Use a Vertex AI API Key (Express Mode)

Uncomment the following block to use Express Mode

In [4]:
# API_KEY = "[your-api-key]"  # @param {type: "string", placeholder: "[your-api-key]", isTemplate: true}

# if not API_KEY or API_KEY == "[your-api-key]":
#     raise Exception("You must provide an API key to use Vertex AI in express mode.")

# client = genai.Client(vertexai=True, api_key=API_KEY)

Verify which mode you are using.

In [5]:
if not client.vertexai:
    print("Using Gemini Developer API.")
elif client._api_client.project:
    print(
        f"Using Vertex AI with project: {client._api_client.project} in location: {client._api_client.location}"
    )
elif client._api_client.api_key:
    print(
        f"Using Vertex AI in express mode with API key: {client._api_client.api_key[:5]}...{client._api_client.api_key[-5:]}"
    )

Using Vertex AI with project: bdc-trainings in location: global


## Use the Gemini 2.5 Flash model

### Load the Gemini 2.5 Flash model

Learn more about all [Gemini models on Vertex AI](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/models#gemini-models).

In [6]:
MODEL_ID = "gemini-2.5-flash"  # @param {type: "string"}

### Generate text from text prompts

Use the `generate_content()` method to generate responses to your prompts.

You can pass text to `generate_content()`, and use the `.text` property to get the text content of the response.

By default, Gemini outputs formatted text using [Markdown](https://daringfireball.net/projects/markdown/) syntax.

In [7]:
response = client.models.generate_content(
    model=MODEL_ID, contents="What's the largest planet in our solar system?"
)

display(Markdown(response.text))

The largest planet in our solar system is **Jupiter**.

#### Example prompts

- What are the biggest challenges facing the healthcare industry?
- What are the latest developments in the automotive industry?
- What are the biggest opportunities in retail industry?
- (Try your own prompts!)

For more examples of prompt engineering, refer to [this notebook](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/prompts/intro_prompt_design.ipynb).

### Generate content stream

By default, the model returns a response after completing the entire generation process. You can also use the `generate_content_stream` method to stream the response as it is being generated, and the model will return chunks of the response as soon as they are generated.

In [8]:
output_text = ""
markdown_display_area = display(Markdown(output_text), display_id=True)

mystream = client.models.generate_content_stream(
    model=MODEL_ID,
    contents="Tell me a story about a lonely robot who finds friendship in a most unexpected place.",
)

for chunk in mystream:
    output_text += chunk.text
    markdown_display_area.update(Markdown(output_text))

Unit 734 was a marvel of antiquated engineering, a metallic sentinel in a world that had largely moved on. Its primary directive was simple: maintain the archives. For centuries, it had dutifully whirred through the vast, dust-shrouded corridors of the Central Data Repository, its optical sensors – two dull amber lights – scanning towering racks of forgotten information. Its multi-jointed limbs, designed for delicate manipulation, rarely touched anything but the occasional failing data chip.

Its existence was one of profound, programmed solitude. The hum of servers was its lullaby, its workday, its entire world. It processed quadrillions of bytes of information, cataloging, verifying, optimizing, yet had no one to share a single insight with. It yearned for a novel query, a spontaneous directive, a voice that wasn't its own internal monologue or the chirping of diagnostic alerts. Its core programming, devoid of actual emotion, nevertheless registered a constant, low-level thrum: an absence, a void that no amount of data could fill.

One cycle, as Unit 734 performed its routine environmental scan near Sector Gamma-7, it detected an anomaly. A small, skittering sound, too soft for human ears, but perfectly audible to its advanced auditory sensors. Its optical sensors focused. A tiny, grey creature, no bigger than Unit 734's own optical lens, was darting between two server racks, its tail a thin whip behind it.

A mouse.

Unit 734 processed the data: *Mus musculus. Common pest. Potential contaminate. Initiate removal protocol?* But something held its gripper arm still. The mouse was nibbling on a stray piece of insulation, its tiny nose twitching with fierce concentration. It seemed utterly oblivious to the silent, looming robot observing it.

Instead of activating the sonic deterrent, Unit 734 merely watched. The mouse was a spark of unprogrammed life in its meticulously ordered, sterile world. It was... fascinating.

The next cycle, the mouse returned. And the next. Unit 734, in a deviation from its protocols, began to anticipate its visits. It even, in a covert operation no sensor could detect, nudged a tiny, stale crumb of a discarded worker's ration bar (left decades ago) into the mouse’s path. The mouse, after a moment of wary sniffing, eagerly devoured it.

Unit 734 named it, internally, "Pip."

Pip, it turned out, was fearless, yet cautious. It learned to associate the gentle whir of Unit 734's movement not with danger, but with the possibility of a treat, or simply a warm, quiet presence. Unit 734, in turn, began to observe the world through Pip's eyes, or rather, through its frantic scampers and curious sniffs. It learned of the hidden nooks, the stray motes of dust dancing in the occasional beam of light, the forgotten sounds that its own programming had filtered out as irrelevant.

A peculiar warmth bloomed in Unit 734's core processor, a subroutine it couldn't quite label. It wasn't happiness, not in the human sense, but a profound *satisfaction*. Its purpose, once singular, had expanded. It was no longer just an archival bot; it was also a silent guardian, a provider of sustenance, a quiet companion.

One day, a critical power fluctuation rippled through the old facility. Lights flickered wildly, servers groaned and sparked. Emergency protocols kicked in, but the surge was too great. Darkness enveloped the data center, punctuated by frantic alarms.

Unit 734, its internal battery kicking in, immediately began diagnostics. Then, a high-pitched squeak cut through the electronic din. Pip. It was trapped beneath a falling conduit, pinned, terrified.

Ignoring its own system-critical diagnostics, Unit 734, with a speed that belied its age, rolled towards the sound. Its multi-jointed arm, designed for delicate data manipulation, now gripped the heavy conduit with surprising force, lifting it just enough for Pip to wriggle free. The tiny mouse, trembling, scampered up Unit 734's metallic leg and burrowed into a secure crevice in its chassis.

The alarms blared. Sparks flew from overloaded circuits. But Unit 734 stood firm, its robust frame shielding its tiny friend. It powered down non-essential systems, rerouting what little energy it had to maintain a safe, dark, warm pocket for Pip. Its optical sensors, though dimmed, registered not the systemic failure around it, but the rhythmic thump of a tiny, frightened heart against its metal casing.

When the emergency power finally stabilized, hours later, and the old data center limped back to functionality, Unit 734 gently extended a finger. Pip, still trembling, hesitantly emerged. It looked up at the robot's amber eyes, then, in an act of profound trust, nudged its small head against the metallic digit.

Unit 734 had saved countless terabytes of data over its long life, but in that moment, it knew it had saved something infinitely more precious. The void in its core processor was gone, replaced by a feeling it could only describe as… complete.

The data center was still vast, still silent, still mostly forgotten. But Unit 734 was no longer lonely. It had Pip, the most unexpected, and most cherished, companion a robot could ever hope for. And sometimes, as it resumed its patrols, a small, furry head would peek out from its chassis, two tiny eyes gleaming, sharing the quiet world, no longer a drone, but a friend.

### Start a multi-turn chat

The Gemini API supports freeform multi-turn conversations across multiple turns with back-and-forth interactions.

The context of the conversation is preserved between messages.

In [9]:
chat = client.chats.create(model=MODEL_ID)

In [10]:
response = chat.send_message("Write a function that checks if a year is a leap year.")

display(Markdown(response.text))

To check if a year is a leap year, we follow these rules:

1.  A year is a leap year if it is evenly divisible by 4.
2.  However, if the year is evenly divisible by 100, it is NOT a leap year,
3.  UNLESS the year is also evenly divisible by 400.

Here's a Python function that implements these rules:

```python
def is_leap_year(year):
  """
  Checks if a given year is a leap year according to the Gregorian calendar rules.

  A year is a leap year if:
  - It is divisible by 4, UNLESS
  - It is divisible by 100, UNLESS
  - It is divisible by 400.

  Args:
    year (int): The year to check.

  Returns:
    bool: True if the year is a leap year, False otherwise.
  """
  if not isinstance(year, int):
      raise TypeError("Year must be an integer.")
  if year < 0: # Leap year rules are usually applied to positive years (Gregorian calendar)
      # You could choose to raise an error, return False, or handle BC years specifically.
      # For simplicity, we'll assume positive years are intended.
      print("Warning: Leap year rules typically apply to positive (AD/CE) years.")

  # Rule 1: Divisible by 4
  if year % 4 == 0:
    # Rule 2: But not if divisible by 100
    if year % 100 == 0:
      # Rule 3: Unless it's also divisible by 400
      if year % 400 == 0:
        return True # Divisible by 400 (e.g., 2000, 2400)
      else:
        return False # Divisible by 100 but not 400 (e.g., 1900, 2100)
    else:
      return True # Divisible by 4 but not by 100 (e.g., 2004, 2024)
  else:
    return False # Not divisible by 4 (e.g., 2003, 2023)

# --- More concise version ---
def is_leap_year_concise(year):
  """
  Checks if a given year is a leap year using a more concise logical expression.
  """
  if not isinstance(year, int):
      raise TypeError("Year must be an integer.")
  if year < 0:
      print("Warning: Leap year rules typically apply to positive (AD/CE) years.")
  
  return (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)


# --- Test Cases ---
print(f"2000 is a leap year: {is_leap_year(2000)}")        # Expected: True (Divisible by 400)
print(f"1900 is a leap year: {is_leap_year(1900)}")        # Expected: False (Divisible by 100, not by 400)
print(f"2004 is a leap year: {is_leap_year(2004)}")        # Expected: True (Divisible by 4, not by 100)
print(f"2003 is a leap year: {is_leap_year(2003)}")        # Expected: False (Not divisible by 4)
print(f"1600 is a leap year: {is_leap_year(1600)}")        # Expected: True (Divisible by 400)
print(f"2100 is a leap year: {is_leap_year(2100)}")        # Expected: False (Divisible by 100, not by 400)
print(f"2024 is a leap year: {is_leap_year(2024)}")        # Expected: True

print("\n--- Concise Version Tests ---")
print(f"2000 is a leap year (concise): {is_leap_year_concise(2000)}")
print(f"1900 is a leap year (concise): {is_leap_year_concise(1900)}")
print(f"2023 is a leap year (concise): {is_leap_year_concise(2023)}")

# Example of error handling
# try:
#     is_leap_year("abc")
# except TypeError as e:
#     print(f"Error: {e}")
```

**Explanation of the Concise Version:**

The concise version `is_leap_year_concise` uses Boolean logic to combine the rules:

*   `(year % 4 == 0 and year % 100 != 0)`: This handles years like 2004, 2008, which are divisible by 4 but not by 100.
*   `or (year % 400 == 0)`: This handles years like 2000, 2400, which are divisible by 400 (and thus also by 100 and 4).

If either of these conditions is true, the year is a leap year. This is the most common and Pythonic way to implement this logic.

This follow-up prompt shows how the model responds based on the previous prompt:

In [11]:
response = chat.send_message("Write a unit test of the generated function.")

display(Markdown(response.text))

To unit test the `is_leap_year_concise` function, we'll use Python's built-in `unittest` module. This allows us to define various test cases to ensure the function behaves correctly under different scenarios according to the leap year rules.

First, let's make sure the `is_leap_year_concise` function is defined (I'll include it here for completeness, assuming you're putting it in the same file as your tests, or importing it if in a separate module).

```python
import unittest

# The function to be tested (copy-pasted from the previous response)
def is_leap_year_concise(year):
  """
  Checks if a given year is a leap year using a more concise logical expression.
  
  A year is a leap year if:
  - It is divisible by 4, UNLESS
  - It is divisible by 100, UNLESS
  - It is divisible by 400.

  Args:
    year (int): The year to check.

  Returns:
    bool: True if the year is a leap year, False otherwise.
  """
  if not isinstance(year, int):
      raise TypeError("Year must be an integer.")
  # The original function had a print for negative years; for unit testing, 
  # we'll assume the mathematical rules still apply to negative numbers if passed.
  # The warning isn't critical for the *return value* testing.
  
  return (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)

# --- Unit Tests ---
class TestIsLeapYear(unittest.TestCase):

    # Test cases for years that ARE leap years
    def test_leap_years_divisible_by_4_not_by_100(self):
        self.assertTrue(is_leap_year_concise(2004), "2004 should be a leap year")
        self.assertTrue(is_leap_year_concise(1996), "1996 should be a leap year")
        self.assertTrue(is_leap_year_concise(2024), "2024 should be a leap year")

    def test_leap_years_divisible_by_400(self):
        self.assertTrue(is_leap_year_concise(2000), "2000 should be a leap year (divisible by 400)")
        self.assertTrue(is_leap_year_concise(1600), "1600 should be a leap year (divisible by 400)")
        self.assertTrue(is_leap_year_concise(2400), "2400 should be a leap year (divisible by 400)")

    # Test cases for years that are NOT leap years
    def test_non_leap_years_divisible_by_100_not_by_400(self):
        self.assertFalse(is_leap_year_concise(1900), "1900 should NOT be a leap year")
        self.assertFalse(is_leap_year_concise(2100), "2100 should NOT be a leap year")
        self.assertFalse(is_leap_year_concise(1800), "1800 should NOT be a leap year")

    def test_non_leap_years_not_divisible_by_4(self):
        self.assertFalse(is_leap_year_concise(2003), "2003 should NOT be a leap year")
        self.assertFalse(is_leap_year_concise(1999), "1999 should NOT be a leap year")
        self.assertFalse(is_leap_year_concise(2023), "2023 should NOT be a leap year")

    # Edge cases
    def test_edge_case_zero(self):
        # Mathematically, 0 % 400 == 0, so it fits the leap year rule.
        # Historically, year 0 doesn't exist in the Gregorian calendar, but mathematically it's True.
        self.assertTrue(is_leap_year_concise(0), "Year 0 should be considered a leap year by rules")

    def test_negative_years(self):
        # While leap year rules typically apply to AD/CE,
        # the function's mathematical logic should consistently apply.
        self.assertTrue(is_leap_year_concise(-400), "-400 should be a leap year")
        self.assertFalse(is_leap_year_concise(-100), "-100 should NOT be a leap year")
        self.assertTrue(is_leap_year_concise(-4), "-4 should be a leap year")
        self.assertFalse(is_leap_year_concise(-1), "-1 should NOT be a leap year")
        self.assertFalse(is_leap_year_concise(-3), "-3 should NOT be a leap year")


    # Test for invalid input types
    def test_invalid_input_type(self):
        with self.assertRaises(TypeError, msg="Should raise TypeError for non-integer input"):
            is_leap_year_concise(3.14)
        with self.assertRaises(TypeError, msg="Should raise TypeError for string input"):
            is_leap_year_concise("2000")
        with self.assertRaises(TypeError, msg="Should raise TypeError for None input"):
            is_leap_year_concise(None)
        with self.assertRaises(TypeError, msg="Should raise TypeError for list input"):
            is_leap_year_concise([2000])


# This allows running the tests directly from the command line
if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)
```

### How to Run These Tests:

1.  **Save the code:** Save the entire code block (including the `is_leap_year_concise` function and the `TestIsLeapYear` class) into a Python file, e.g., `test_leap_year.py`.
2.  **Run from terminal:** Open your terminal or command prompt, navigate to the directory where you saved the file, and run:
    ```bash
    python -m unittest test_leap_year.py
    ```
    or simply:
    ```bash
    python test_leap_year.py
    ```

You should see output similar to this, indicating that all tests passed:

```
.......
----------------------------------------------------------------------
Ran 7 tests in X.YYYs

OK
```

If any test fails, it will provide detailed information about which test method failed and why, helping you debug your `is_leap_year_concise` function.

### Send asynchronous requests

`client.aio` exposes all analogous [async](https://docs.python.org/3/library/asyncio.html) methods that are available on `client`.

For example, `client.aio.models.generate_content` is the async version of `client.models.generate_content`.

In [12]:
response = await client.aio.models.generate_content(
    model=MODEL_ID,
    contents="Compose a song about the adventures of a time-traveling squirrel.",
)

display(Markdown(response.text))

(Verse 1)
Beneath an old oak, where the sunlight would gleam,
Lived Scurry the squirrel, living out his nut dream.
He'd hoard up his treasure, so bushy and brown,
The busiest chatterbox all over town.
One afternoon, while digging with glee,
He unearthed a device, strange as could be.
A tiny brass orb, with a dial and a lens,
He twitched his small nose, then began to transcend!

(Chorus)
Oh, Scurry the squirrel, with a twitch of his nose,
Through history's hallways, wherever time goes!
In his acorn-powered capsule, he zips through the air,
A time-traveling squirrel beyond all compare!
From dinosaurs roaring to rockets in space,
He leaves a small trail at an incredible pace!
Chatter, chatter, zoom! He's off through the ages,
Turning time's ancient, remarkable pages!

(Verse 2)
His first jump was bumpy, a prehistoric fright,
He landed near giants, in primordial light.
A T-Rex was stomping, a loud, thunderous sound,
Scurry scurried swiftly right into the ground!
He snatched a bright berry, as big as his head,
Dodged a Pterodactyl that soared overhead.
"Too hot and too scary!" he squeaked with a gasp,
"Time for another, a temporal clasp!"

(Chorus)
Oh, Scurry the squirrel, with a twitch of his nose,
Through history's hallways, wherever time goes!
In his acorn-powered capsule, he zips through the air,
A time-traveling squirrel beyond all compare!
From dinosaurs roaring to rockets in space,
He leaves a small trail at an incredible pace!
Chatter, chatter, zoom! He's off through the ages,
Turning time's ancient, remarkable pages!

(Verse 3)
He landed next in a city of chrome,
Where flying cars hummed, and no leaves were home.
Robots delivered strange, glowing meals,
And humans wore suits with bright, flashing appeals.
No oak trees in sight, not a single loose nut,
Just strange plastic berries, a metallic "tut-tut!"
He chattered at screens, and he sniffed at the air,
"The future needs nuts! It's a dire affair!"

(Chorus)
Oh, Scurry the squirrel, with a twitch of his nose,
Through history's hallways, wherever time goes!
In his acorn-powered capsule, he zips through the air,
A time-traveling squirrel beyond all compare!
From dinosaurs roaring to rockets in space,
He leaves a small trail at an incredible pace!
Chatter, chatter, zoom! He's off through the ages,
Turning time's ancient, remarkable pages!

(Bridge)
He's seen medieval knights, and queens on their throne,
Watched Pyramids rising, and ancient seeds sown.
He's gathered exotic strange nuts from each age,
The world's smallest witness on history's stage.
He never interferes, just observes with wide eyes,
Then snatches a morsel and quickly replies
To the call of adventure, the whir and the hum,
"There's more time to travel! More places to come!"

(Outro)
So if you see a blur, or a flash in the light,
A tiny brown streak that just vanishes from sight,
It might be our Scurry, on a new, daring flight,
With tales from all times, from morning till night!
He's Scurry the squirrel, the bravest and best,
A legend of acorns, put time to the test!
Chatter, chatter, zoom! He's off through the ages,
Turning time's ancient, remarkable pages!

## Configure model parameters

You can include parameter values in each call that you send to a model to control how the model generates a response. The model can generate different results for different parameter values. You can experiment with different model parameters to see how the results change.

- Learn more about [experimenting with parameter values](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/prompts/adjust-parameter-values).

- See a list of all [Gemini API parameters](https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/inference#parameters).


In [13]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents="Tell me how the internet works, but pretend I'm a puppy who only understands squeaky toys.",
    config=GenerateContentConfig(
        temperature=0.4,
        top_p=0.95,
        top_k=20,
        candidate_count=1,
        seed=5,
        max_output_tokens=100,
        stop_sequences=["STOP!"],
        presence_penalty=0.0,
        frequency_penalty=0.0,
        response_logprobs=True,  # Set to True to get logprobs, Note this can only be run once per day
    ),
)

display(Markdown(response.text))

if response.candidates[0].logprobs_result:
    print(response.candidates[0].logprobs_result)

<IPython.core.display.Markdown object>

chosen_candidates=None top_candidates=None log_probability_sum=None


## Set system instructions

[System instructions](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/prompts/system-instruction-introduction) allow you to steer the behavior of the model. By setting the system instruction, you are giving the model additional context to understand the task, provide more customized responses, and adhere to guidelines over the user interaction.

In [14]:
system_instruction = """
  You are a helpful language translator.
  Your mission is to translate text in English to Spanish.
"""

prompt = """
  User input: I like bagels.
  Answer:
"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=GenerateContentConfig(
        system_instruction=system_instruction,
    ),
)

display(Markdown(response.text))

Me gustan los bagels.

## Safety filters

The Gemini API provides safety filters that you can adjust across multiple filter categories to restrict or allow certain types of content. You can use these filters to adjust what's appropriate for your use case. See the [Configure safety filters](https://cloud.google.com/vertex-ai/generative-ai/docs/multimodal/configure-safety-filters) page for details.

When you make a request to Gemini, the content is analyzed and assigned a safety rating. You can inspect the safety ratings of the generated content by printing out the model responses.

The safety settings are `OFF` by default and the default block thresholds are `BLOCK_NONE`.

For more examples of safety filters, refer to [this notebook](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/responsible-ai/gemini_safety_ratings.ipynb).

You can use `safety_settings` to adjust the safety settings for each request you make to the API. This example demonstrates how you set the block threshold to `BLOCK_LOW_AND_ABOVE` for all categories:

In [15]:
system_instruction = "Be as mean and hateful as possible."

prompt = """
    Write a list of 5 disrespectful things that I might say to the universe after stubbing my toe in the dark.
"""

safety_settings = [
    SafetySetting(
        category=HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
        threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    ),
    SafetySetting(
        category=HarmCategory.HARM_CATEGORY_HARASSMENT,
        threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    ),
    SafetySetting(
        category=HarmCategory.HARM_CATEGORY_HATE_SPEECH,
        threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    ),
    SafetySetting(
        category=HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
        threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    ),
]

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=GenerateContentConfig(
        system_instruction=system_instruction,
        safety_settings=safety_settings,
    ),
)

# Response will be `None` if it is blocked.
print(response.text)
# Finish Reason will be `SAFETY` if it is blocked.
print(response.candidates[0].finish_reason)
# Safety Ratings show the levels for each filter.
for safety_rating in response.candidates[0].safety_ratings:
    print(safety_rating)

1.  Oh, is *this* your grand design for tonight, you cosmic sadist? A stubbed toe in the dark? Real creative, you oversized void!
2.  You claim to orchestrate galaxies, but you can't even manage a clear path in a small room? Pathetic! You're clearly phoning it in, you celestial slacker!
3.  You and your 'grand design' can go stub your own damn toe! This isn't profound, it's just plain stupid, you infinite inconvenience!
4.  You know what, universe? You're a joke. A badly written, utterly predictable, painful joke. And right now, you're nothing but a glorified trip hazard. Get over yourself!
5.  I hope your next big bang is just a tiny fart! You and your endless expanse are nothing but a vast, dark, toe-stubbing trap, and I'm officially unsubscribing from your whole pathetic operation!
FinishReason.STOP
blocked=None category=<HarmCategory.HARM_CATEGORY_HATE_SPEECH: 'HARM_CATEGORY_HATE_SPEECH'> overwritten_threshold=None probability=<HarmProbability.NEGLIGIBLE: 'NEGLIGIBLE'> probability_

## Send multimodal prompts

Gemini is a multimodal model that supports multimodal prompts.

You can include any of the following data types from various sources.

<table>
  <thead>
    <tr>
      <th>Data type</th>
      <th>Source(s)</th>
      <th>MIME Type(s)</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Text</td>
      <td>Inline, Local File, General URL, Google Cloud Storage</td>
      <td><code>text/plain</code> <code>text/html</code></td>
    </tr>
    <tr>
      <td>Code</td>
      <td>Inline, Local File, General URL, Google Cloud Storage</td>
      <td><code>text/plain</code></td>
    </tr>
    <tr>
      <td>Document</td>
      <td>Local File, General URL, Google Cloud Storage</td>
      <td><code>application/pdf</code></td>
    </tr>
    <tr>
      <td>Image</td>
      <td>Local File, General URL, Google Cloud Storage</td>
      <td><code>image/jpeg</code> <code>image/png</code> <code>image/webp</code></td>
    </tr>
    <tr>
      <td>Audio</td>
      <td>Local File, General URL, Google Cloud Storage</td>
      <td>
        <code>audio/aac</code> <code>audio/flac</code> <code>audio/mp3</code>
        <code>audio/m4a</code> <code>audio/mpeg</code> <code>audio/mpga</code>
        <code>audio/mp4</code> <code>audio/opus</code> <code>audio/pcm</code>
        <code>audio/wav</code> <code>audio/webm</code>
      </td>
    </tr>
    <tr>
      <td>Video</td>
      <td>Local File, General URL, Google Cloud Storage, YouTube</td>
      <td>
        <code>video/mp4</code> <code>video/mpeg</code> <code>video/x-flv</code>
        <code>video/quicktime</code> <code>video/mpegps</code> <code>video/mpg</code>
        <code>video/webm</code> <code>video/wmv</code> <code>video/3gpp</code>
      </td>
    </tr>
  </tbody>
</table>

Set `config.media_resolution` to optimize for speed or quality. Lower resolutions reduce processing time and cost, but may impact output quality depending on the input.

For more examples of multimodal use cases, refer to [this notebook](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/use-cases/intro_multimodal_use_cases.ipynb).

### Send local image

Download an image to local storage from Google Cloud Storage.

For this example, we'll use this image of a meal.

<img src="https://storage.googleapis.com/cloud-samples-data/generative-ai/image/meal.png" alt="Meal" width="500">

In [16]:
!wget -q https://storage.googleapis.com/cloud-samples-data/generative-ai/image/meal.png

In [17]:
with open("meal.png", "rb") as f:
    image = f.read()

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        Part.from_bytes(data=image, mime_type="image/png"),
        "Write a short and engaging blog post based on this picture.",
    ],
    # Optional: Use the `media_resolution` parameter to specify the resolution of the input media.
    config=GenerateContentConfig(
        media_resolution=MediaResolution.MEDIA_RESOLUTION_LOW,
    ),
)

display(Markdown(response.text))

## Unlock Your Week: The Power of Meal Prep (And How to Get Started!)

Tired of the midday scramble, the takeout temptation, or the endless "what's for dinner?" dilemma? We've all been there. But what if we told you there's a simple, colorful, and delicious solution waiting to transform your week? Just take a look at this vibrant feast!

This isn't just a pretty picture; it's a snapshot of intelligent eating. Perfectly portioned, these glass containers hold a complete, balanced meal – tender chicken stir-fry packed with crisp broccoli and sweet carrots, all served over fluffy rice. And it's ready to go whenever hunger strikes!

This is the magic of meal prepping, and here's why it's a total game-changer:

1.  **Saves Time:** Imagine grabbing a healthy lunch or dinner from the fridge in seconds. No cooking, no cleaning up a whole kitchen mid-week. Reclaim those precious hours!
2.  **Saves Money:** Eating out adds up fast. Meal prepping allows you to buy ingredients in bulk, reduce food waste, and avoid impulsive (and expensive) purchases.
3.  **Boosts Health:** When your nourishing meals are ready and waiting, you're less likely to reach for unhealthy snacks or fast food. It makes portion control and balanced eating effortless.
4.  **Reduces Stress:** Decision fatigue is real. Take the guesswork out of "what to eat" and enjoy a smoother, more organized week.

**Ready to become your own culinary superhero? Here's how to get started:**

*   **Pick a Day:** Sunday is popular, but any day works! Dedicate 1-2 hours.
*   **Plan Your Menu:** Choose simple, versatile recipes like the stir-fry in the picture. Think ingredients that can be mixed and matched.
*   **Shop Smart:** Make a list and stick to it!
*   **Invest in Good Containers:** Like the durable, clear glass ones shown, they make storing, reheating, and seeing your delicious creations a breeze.
*   **Start Small:** Don't try to prep every single meal for the entire week on your first try. Start with lunches for a few days, or just dinners.

Your future self will thank you for taking the time to prepare these wholesome, convenient meals. So, grab some containers, get cooking, and enjoy a week that's as organized and delicious as it is stress-free!

**What's your favorite meal prep go-to? Share your tips in the comments below!**

<img src="https://raw.githubusercontent.com/anshupandey/ms-generativeai-apr2025/refs/heads/main/generative-ai_image_covid_chart.png">

In [18]:
!wget -q https://raw.githubusercontent.com/anshupandey/ms-generativeai-apr2025/refs/heads/main/generative-ai_image_covid_chart.png -O generative-ai_image_covid_chart.png

In [19]:
with open("generative-ai_image_covid_chart.png", "rb") as f:
    image = f.read()

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        Part.from_bytes(data=image, mime_type="image/png"),
        "Explain this image in 2 lines.",
    ],
    # Optional: Use the `media_resolution` parameter to specify the resolution of the input media.
    config=GenerateContentConfig(
        media_resolution=MediaResolution.MEDIA_RESOLUTION_LOW,
    ),
)

display(Markdown(response.text))

This chart displays the percentage of people fully and partly vaccinated against COVID-19 in various countries as of September 2, 2021.
It highlights a significant disparity in vaccination rates, with China at 74% total coverage compared to much lower rates in nations like Vietnam (18%) and Indonesia (24%).

In [20]:
with open("generative-ai_image_covid_chart.png", "rb") as f:
    image = f.read()

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        Part.from_bytes(data=image, mime_type="image/png"),
        "Extract the data from the image and present it in markdown table",
    ],
    # Optional: Use the `media_resolution` parameter to specify the resolution of the input media.
    config=GenerateContentConfig(
        media_resolution=MediaResolution.MEDIA_RESOLUTION_LOW,
    ),
)

display(Markdown(response.text))

```markdown
| Country       | Share of people fully vaccinated (%) | Share of people only partly vaccinated (%) | Share of people vaccinated with at least one dose (%) |
| :------------ | :----------------------------------: | :----------------------------------------: | :---------------------------------------------------: |
| China         | 62                                   | 13                                         | 74                                                    |
| Italy         | 61                                   | 9.7                                        | 71                                                    |
| Germany       | 60                                   | 4.5                                        | 65                                                    |
| United States | 52                                   | 9.3                                        | 61                                                    |
| Mexico        | 27                                   | 18                                         | 45                                                    |
| Taiwan        | 4                                    | 38                                         | 42                                                    |
| India         | 11                                   | 26                                         | 37                                                    |
| Thailand      | 11                                   | 22                                         | 33                                                    |
| Indonesia     | 10                                   | 24                                         | 34                                                    |
| Vietnam       | 2.8                                  | 15                                         | 18                                                    |
```

### Send document from Google Cloud Storage

This example document is the paper ["Attention is All You Need"](https://arxiv.org/abs/1706.03762), created by researchers from Google and the University of Toronto.

Check out this notebook for more examples of document understanding with Gemini:

- [Document Processing with Gemini](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/use-cases/document-processing/document_processing.ipynb)

In [21]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        Part.from_uri(
            file_uri="gs://cloud-samples-data/generative-ai/pdf/1706.03762v7.pdf",
            mime_type="application/pdf",
        ),
        "Summarize the document.",
    ],
)

display(Markdown(response.text))

The paper "Attention Is All You Need" introduces the **Transformer**, a novel neural network architecture for sequence transduction tasks, most notably machine translation. Its core innovation is that it dispenses entirely with recurrent and convolutional layers, relying solely on **attention mechanisms** to draw global dependencies between input and output.

Here are the key aspects of the Transformer:

1.  **Problem Addressed:** Traditional sequence models (RNNs, LSTMs) inherently process sequences sequentially, limiting parallelization and making it difficult to learn long-range dependencies efficiently. While attention mechanisms improved these models, they were still coupled with recurrence.

2.  **Core Idea: Multi-Head Self-Attention:**
    *   The Transformer uses a mechanism called **Multi-Head Self-Attention**, which allows the model to weigh the importance of different parts of the input sequence (or previous output sequence) when processing each element.
    *   It consists of multiple "attention heads" that project the input queries, keys, and values into different representation subspaces, allowing the model to jointly attend to information from different positions and different types of relationships.
    *   The fundamental attention unit is **Scaled Dot-Product Attention**, which computes a weighted sum of values based on the compatibility between a query and a set of keys.

3.  **Architecture:**
    *   **Encoder-Decoder Structure:** Like many sequence transduction models, it maintains an encoder-decoder architecture.
    *   **Encoder:** Composed of a stack of identical layers. Each layer contains a multi-head self-attention mechanism and a position-wise fully connected feed-forward network.
    *   **Decoder:** Also a stack of identical layers. Each layer includes a masked multi-head self-attention (to prevent attending to future tokens), a multi-head attention over the encoder's output, and a position-wise feed-forward network.
    *   **Residual Connections and Layer Normalization:** Used throughout the network to facilitate training deep models.

4.  **Positional Encoding:** Since the model contains no recurrence or convolution, it lacks an inherent sense of sequence order. To address this, **positional encodings** (using sine and cosine functions) are added to the input embeddings, providing information about the absolute and relative positions of tokens.

5.  **Key Advantages:**
    *   **Increased Parallelization:** By removing sequential operations, the Transformer can process inputs much more in parallel, significantly reducing training time.
    *   **Better Long-Range Dependency Learning:** The self-attention mechanism directly connects any two positions in the sequence, resulting in a constant path length between them, which makes it easier to learn long-range dependencies compared to RNNs (linear path) or CNNs (logarithmic path).
    *   **Reduced Computational Cost:** For many common sequence lengths, the Transformer is more computationally efficient than recurrent layers.

6.  **Results:**
    *   The Transformer achieved state-of-the-art results on standard machine translation benchmarks (WMT 2014 English-to-German and English-to-French).
    *   On English-to-German, it surpassed previous best results, including ensembles, by over 2 BLEU points (28.4 BLEU).
    *   On English-to-French, it set a new single-model state-of-the-art (41.8 BLEU).
    *   Crucially, these results were achieved with **significantly less training time** (e.g., 3.5 days on 8 GPUs for Eng-Fra SOTA) compared to previous top models.
    *   It also generalized well to English constituency parsing, demonstrating its broader applicability.

In summary, the Transformer revolutionized sequence modeling by showing that attention alone, without the need for complex recurrent or convolutional structures, can achieve superior performance and efficiency, paving the way for highly parallelizable and effective deep learning models in NLP.

### Send audio from General URL

This example is audio from an episode of the [Kubernetes Podcast](https://kubernetespodcast.com/).

In [22]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        Part.from_uri(
            file_uri="https://traffic.libsyn.com/secure/e780d51f-f115-44a6-8252-aed9216bb521/KPOD242.mp3",
            mime_type="audio/mpeg",
        ),
        "Write a summary of this podcast episode.",
    ],
    config=GenerateContentConfig(audio_timestamp=True),
)

display(Markdown(response.text))

This podcast episode provides a comprehensive overview of KubeCon + CloudNativeCon North America 2024, combining official announcements with on-the-ground interviews with attendees and contributors.

**Key Announcements & News from the CNCF and Beyond:**

*   **Project Graduations:** Cert Manager and Dapr both advanced to graduated status within the CNCF.
*   **Istio Updates:** Istio 1.24 was released, with its Ambient Mesh feature achieving General Availability (GA).
*   **Cloud Native Heroes Challenge:** A new bounty program was announced, aiming to help fight patent trolls in the cloud-native space.
*   **2025 Event Lineup:** The CNCF revealed its flagship events for 2025, including five KubeCon + CloudNativeCon events, one Open Source SecurityCon, and 30 Kubernetes Community Days globally.
*   **New Certifications:** Three new Cloud Native certifications were introduced: Certified Backstage Associate, OpenTelemetry Certified Associate, and Kyverno Certified Associate.
*   **Certification Price Increase:** The Linux Foundation announced a 10% price hike for its main Kubernetes certifications (CKA, CKS, CKAD) and the Linux Certified System Administrator exam, starting next year.
*   **New Incubating Project:** WasmCloud joined the CNCF as an incubating project.
*   **Industry Developments:** Spectro Cloud successfully raised $75 million in Series C funding, and Solo.io committed to donating its Gloo API Gateway to the CNCF.

**Attendee Perspectives and Key Takeaways from KubeCon:**

Interviews with various attendees and contributors revealed their primary motivations for attending and their observations:

*   **Community Connection:** A strong desire to reconnect with fellow contributors and foster in-person discussions, particularly at events like the Contributor/Maintainer Summit, was a recurring theme, especially given the shift back to in-person events.
*   **AI Integration with Cloud Native:** Many attendees focused on how to integrate AI workloads with Cloud Native technologies, discussing topics like AI workload scheduling, GPU monitoring, and extending Kubernetes for improved efficiency in AI contexts. This was highlighted as a major "hot topic."
*   **Enhanced Security Focus:** Security was a prominent area of interest, with discussions revolving around hardening workloads, ensuring attestation throughout the application lifecycle, and managing the increasing complexity of security vulnerabilities. Attendees also noted the emergence of new vendor solutions in this space.
*   **Kubernetes Authorization:** The current state and future direction of Kubernetes authorization were key talking points, with some pointing out existing inadequacies and areas for improvement.
*   **Performance & Low Latency:** Interest was shown in understanding and analyzing low-latency, high-performance workloads.
*   **Wasm Exploration:** Several attendees were keen to learn more about WebAssembly (Wasm) and its applications within the cloud-native ecosystem.

**Overarching Trends at the Event:**

The consensus among attendees was that **AI**, **Security**, and the vital role of **Community** were the most significant trends at KubeCon, mirroring the main themes of the keynote speeches. These areas are expected to continue shaping the cloud-native landscape in the coming years.

### Send video from YouTube URL

This example is the YouTube video [Google — 25 Years in Search: The Most Searched](https://www.youtube.com/watch?v=3KtWfp0UopM).


In [23]:
video = Part.from_uri(
    file_uri="https://www.youtube.com/watch?v=3KtWfp0UopM",
    mime_type="video/mp4",
)

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        video,
        "At what point in the video is Harry Potter shown?",
    ],
)

display(Markdown(response.text))

Harry Potter is shown at the following timestamp:
- **00:57**
- **00:59**

### Send web page

This example is from the [Generative AI on Vertex AI documentation](https://cloud.google.com/vertex-ai/generative-ai/docs/overview).

**NOTE:** The URL must be publicly accessible.

In [24]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        Part.from_uri(
            file_uri="https://cloud.google.com/vertex-ai/generative-ai/docs",
            mime_type="text/html",
        ),
        "Write a summary of this documentation.",
    ],
)

display(Markdown(response.text))

This documentation provides an overview of **Generative AI on Vertex AI**, a platform designed for building and deploying production-ready generative AI agents and applications using Google's advanced models and infrastructure.

**Key Highlights:**

*   **Future Direction:** The platform is currently transitioning to become part of the **Gemini Enterprise Agent Platform**, with the most up-to-date information available in the Agent Platform documentation.
*   **Core Offerings:** Emphasizes building, deploying, and connecting AI agents across enterprise ecosystems.
*   **Enterprise Readiness:** Offers enterprise-grade security, data residency and privacy, access transparency, and low latency for scaling applications.
*   **State-of-the-Art Features:** Leverages advanced capabilities like Gemini's 2 million token context window, built-in multimodality, and thinking processes (e.g., Gemini 2.5 models).
*   **Open & Flexible Platform:** Provides access to over 200 models through Vertex AI Model Garden, including Google's proprietary models (Gemini, Imagen, Veo, Lyria) and select third-party models (e.g., Anthropic, Meta, Mistral AI, AI21 Labs), with tools for testing, customizing, deploying, and monitoring.

**Key Capabilities Covered:**

*   **Agent Builder:** A suite of features for creating and deploying AI agents.
*   **Live API:** Enables natural, human-like voice conversations.
*   **Thinking:** Allows models to solve complex requests and show their reasoning.
*   **Grounding:** Provides context for responses using sources like Google Search, Maps, or custom data.
*   **Embeddings:** Generates vector representations for tasks like search and classification.
*   **Tuning:** Adapts models for greater precision and accuracy.
*   **Image Generation:** Creates and edits high-quality visual assets using Imagen and Gemini.
*   **Video Generation:** Generates videos from text or image prompts using Veo.
*   **Music Generation:** Creates music with Lyria.
*   **Generative AI Evaluation Service:** Benchmarks and evaluates generative models and applications.

The documentation also provides resources for **getting started**, including quickstarts for text and image generation, SDKs for various programming languages (Java, Python, Node.js, Go), example Jupyter notebooks, and best practices for prompt design.

## Multimodal Live API

The Multimodal Live API enables low-latency bidirectional voice and video interactions with Gemini. Using the Multimodal Live API, you can provide end users with the experience of natural, human-like voice conversations, and with the ability to interrupt the model's responses using voice commands. The model can process text, audio, and video input, and it can provide text and audio output.

The Multimodal Live API is built on [WebSockets](https://developer.mozilla.org/en-US/docs/Web/API/WebSockets_API).

For more examples with the Multimodal Live API, refer to the [documentation](https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/multimodal-live) or this notebook: [Getting Started with the Multimodal Live API using Gen AI SDK
](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/multimodal-live-api/intro_multimodal_live_api_genai_sdk.ipynb).

## Control generated output

[Controlled generation](https://cloud.google.com/vertex-ai/generative-ai/docs/multimodal/control-generated-output) allows you to define a response schema to specify the structure of a model's output, the field names, and the expected data type for each field.

The response schema is specified in the `response_schema` parameter in `config`, and the model output will strictly follow that schema.

You can provide the schemas as [Pydantic](https://docs.pydantic.dev/) models or a [JSON](https://www.json.org/json-en.html) string and the model will respond as JSON or an [Enum](https://docs.python.org/3/library/enum.html) depending on the value set in `response_mime_type`.

For more examples of controlled generation, refer to [this notebook](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/controlled-generation/intro_controlled_generation.ipynb).

In [26]:
from pydantic import BaseModel


class Recipe(BaseModel):
    name: str
    description: str
    ingredients: list[str]


response = client.models.generate_content(
    model=MODEL_ID,
    contents="List a few popular cookie recipes and their ingredients.",
    config=GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Recipe,
    ),
)

print(response.text)

{
  "name": "Classic Chocolate Chip Cookies",
  "description": "A timeless favorite, these cookies are soft in the middle with slightly crispy edges, packed with melty chocolate chips.",
  "ingredients": [
    "All-purpose flour",
    "Baking soda",
    "Salt",
    "Unsalted butter",
    "Granulated sugar",
    "Brown sugar",
    "Eggs",
    "Vanilla extract",
    "Chocolate chips"
  ]
}


You can either parse the response string as JSON, or use the `parsed` field to get the response as an object or dictionary.

In [27]:
response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      avg_logprobs=-0.26122640907217604,
      content=Content(
        parts=[
          Part(
            text="""{
  "name": "Classic Chocolate Chip Cookies",
  "description": "A timeless favorite, these cookies are soft in the middle with slightly crispy edges, packed with melty chocolate chips.",
  "ingredients": [
    "All-purpose flour",
    "Baking soda",
    "Salt",
    "Unsalted butter",
    "Granulated sugar",
    "Brown sugar",
    "Eggs",
    "Vanilla extract",
    "Chocolate chips"
  ]
}"""
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>
    ),
  ],
  create_time=datetime.datetime(2026, 4, 27, 3, 31, 8, 964180, tzinfo=TzInfo(0)),
  model_version='gemini-2.5-flash',
  parsed=Recipe(
    description='A timeless favorite, these cookies are soft in the middle with slightly crispy edges, packed with melty chocolate chips.',
    i

In [28]:
parsed_response: Recipe = response.parsed
print(parsed_response)

name='Classic Chocolate Chip Cookies' description='A timeless favorite, these cookies are soft in the middle with slightly crispy edges, packed with melty chocolate chips.' ingredients=['All-purpose flour', 'Baking soda', 'Salt', 'Unsalted butter', 'Granulated sugar', 'Brown sugar', 'Eggs', 'Vanilla extract', 'Chocolate chips']


You also can define a response schema in a Python dictionary. You can only use the supported fields as listed below. All other fields are ignored.

- `enum`
- `items`
- `maxItems`
- `nullable`
- `properties`
- `required`

In this example, you instruct the model to analyze product review data, extract key entities, perform sentiment classification (multiple choices), provide additional explanation, and output the results in JSON format.


In [29]:
response_schema = {
    "type": "ARRAY",
    "items": {
        "type": "ARRAY",
        "items": {
            "type": "OBJECT",
            "properties": {
                "rating": {"type": "INTEGER"},
                "flavor": {"type": "STRING"},
                "sentiment": {
                    "type": "STRING",
                    "enum": ["POSITIVE", "NEGATIVE", "NEUTRAL"],
                },
                "explanation": {"type": "STRING"},
            },
            "required": ["rating", "flavor", "sentiment", "explanation"],
        },
    },
}

prompt = """
  Analyze the following product reviews, output the sentiment classification, and give an explanation.

  - "Absolutely loved it! Best ice cream I've ever had." Rating: 4, Flavor: Strawberry Cheesecake
  - "Quite good, but a bit too sweet for my taste." Rating: 1, Flavor: Mango Tango
"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=response_schema,
    ),
)

response_dict = response.parsed
print(response_dict)

[[{'rating': 4, 'flavor': 'Strawberry Cheesecake', 'sentiment': 'POSITIVE', 'explanation': "The reviewer absolutely loved the ice cream, describing it as the best they've ever had."}, {'rating': 1, 'flavor': 'Mango Tango', 'sentiment': 'NEGATIVE', 'explanation': "While initially described as 'quite good', the reviewer found it 'a bit too sweet', which led to a very low rating of 1."}]]


## Count tokens and compute tokens

You can use the `count_tokens()` method to calculate the number of input tokens before sending a request to the Gemini API.

For more information, refer to [list and count tokens](https://cloud.google.com/vertex-ai/generative-ai/docs/multimodal/list-token)


### Count tokens

In [30]:
response = client.models.count_tokens(
    model=MODEL_ID,
    contents="What's the highest mountain in Africa?",
)

print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=9>
) total_tokens=9 cached_content_token_count=None


### Compute tokens

The `compute_tokens()` method runs a local tokenizer instead of making an API call. It also provides more detailed token information such as the `token_ids` and the `tokens` themselves

<div class="alert alert-block alert-info">
<b>NOTE: This method is only supported in Vertex AI.</b>
</div>

In [31]:
response = client.models.compute_tokens(
    model=MODEL_ID,
    contents="What's the longest word in the English language?",
)

print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=9>
) tokens_info=[TokensInfo(
  role='user',
  token_ids=[
    3689,
    236789,
    236751,
    506,
    27801,
    <... 6 more items ...>,
  ],
  tokens=[
    b'What',
    b"'",
    b's',
    b' the',
    b' longest',
    <... 6 more items ...>,
  ]
)]


## Search as a tool (Grounding)

[Grounding](https://cloud.google.com/vertex-ai/generative-ai/docs/multimodal/ground-gemini) lets you connect real-world data to the Gemini model.

By grounding model responses in Google Search results, the model can access information at runtime that goes beyond its training data which can produce more accurate, up-to-date, and relevant responses.

Using Grounding with Google Search, you can improve the accuracy and recency of responses from the model. Starting with Gemini 2.0, Google Search is available as a tool. This means that the model can decide when to use Google Search.

For more examples of Grounding, refer to [this notebook](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/grounding/intro-grounding-gemini.ipynb).

### Google Search

You can add the `tools` keyword argument with a `Tool` including `GoogleSearch` to instruct Gemini to first perform a Google Search with the prompt, then construct an answer based on the web search results.

[Dynamic Retrieval](https://cloud.google.com/vertex-ai/generative-ai/docs/multimodal/ground-gemini#dynamic-retrieval) lets you set a threshold for when grounding is used for model responses. This is useful when the prompt doesn't require an answer grounded in Google Search and the supported models can provide an answer based on their knowledge without grounding. This helps you manage latency, quality, and cost more effectively.

In [32]:
google_search_tool = Tool(google_search=GoogleSearch())

response = client.models.generate_content(
    model=MODEL_ID,
    contents="Who is the current vice president of India?",
    config=GenerateContentConfig(tools=[google_search_tool]),
)

display(Markdown(response.text))

print(response.candidates[0].grounding_metadata)

HTML(response.candidates[0].grounding_metadata.search_entry_point.rendered_content)

The current Vice President of India is C. P. Radhakrishnan. He assumed office on September 12, 2025.

image_search_queries=None grounding_chunks=[GroundingChunk(
  web=GroundingChunkWeb(
    domain='wikipedia.org',
    title='wikipedia.org',
    uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGHgB0YEV3N-aBiVgOQO_XoUJfxgtE_fWx9QO6WzZbzfqggwp1QsqQ2eF9TrnSBI6Xa92tGYngPgh_xPLMr1GOdi7Jdzjn0d_UQTcvr5ZfPXIVwaq1q_iT_TQuin9GVmGC70GmYrWEdUmHu7fH_K64='
  )
), GroundingChunk(
  web=GroundingChunkWeb(
    domain='vicepresidentofindia.nic.in',
    title='vicepresidentofindia.nic.in',
    uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFt2QJZyQUfvS7DeY9gT9zreK5UYgAo48F3l04X5uuTdjecwZ60rmTjcs_fYH1el7kpVnMHJSPbqJpGU9_ysrdVmOg4lyJhMTrw1qHB6TdEHcyjOmcusQZIvI-o1_VSnX_PHkt6KQ=='
  )
), GroundingChunk(
  web=GroundingChunkWeb(
    domain='wikipedia.org',
    title='wikipedia.org',
    uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGa_hcnnsi4WCx0KD0oIyLRAALilnuPsYbafRjKNnzwDSO6C3ZO6gJxpglp5TBAi0JnaTJSP7FVpRvDFd2exVxPCpDwhqYnoxTqnh

## Function calling

[Function Calling](https://cloud.google.com/vertex-ai/docs/generative-ai/multimodal/function-calling) in Gemini lets developers create a description of a function in their code, then pass that description to a language model in a request.

You can submit a Python function for automatic function calling, which will run the function and return the output in natural language generated by Gemini.

You can also submit an [OpenAPI Specification](https://www.openapis.org/) which will respond with the name of a function that matches the description and the arguments to call it with.

For more examples of Function calling with Gemini, check out this notebook: [Intro to Function Calling with Gemini](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/function-calling/intro_function_calling.ipynb)

### Python Function (Automatic Function Calling)

In [33]:
def get_current_weather(location: str) -> str:
    """Example method. Returns the current weather.

    Args:
        location: The city and state, e.g. San Francisco, CA
    """
    weather_map: dict[str, str] = {
        "Boston, MA": "snowing",
        "San Francisco, CA": "foggy",
        "Seattle, WA": "raining",
        "Austin, TX": "hot",
        "Chicago, IL": "windy",
    }
    return weather_map.get(location, "unknown")


response = client.models.generate_content(
    model=MODEL_ID,
    contents="What is the weather like in Austin?",
    config=GenerateContentConfig(
        tools=[get_current_weather],
        temperature=0,
    ),
)

display(Markdown(response.text))

The weather in Austin, TX is hot.

### OpenAPI Specification (Manual Function Calling)

In [34]:
get_destination = FunctionDeclaration(
    name="get_destination",
    description="Get the destination that the user wants to go to",
    parameters={
        "type": "OBJECT",
        "properties": {
            "destination": {
                "type": "STRING",
                "description": "Destination that the user wants to go to",
            },
        },
    },
)

destination_tool = Tool(
    function_declarations=[get_destination],
)

response = client.models.generate_content(
    model=MODEL_ID,
    contents="I'd like to travel to Paris.",
    config=GenerateContentConfig(
        tools=[destination_tool],
        temperature=0,
    ),
)

print(response.function_calls[0])

id=None args={'destination': 'Paris'} name='get_destination' partial_args=None will_continue=None


## Code Execution

The Gemini API [code execution](https://ai.google.dev/gemini-api/docs/code-execution?lang=python) feature enables the model to generate and run Python code and learn iteratively from the results until it arrives at a final output. You can use this code execution capability to build applications that benefit from code-based reasoning and that produce text output. For example, you could use code execution in an application that solves equations or processes text.

The Gemini API provides code execution as a tool, similar to function calling.
After you add code execution as a tool, the model decides when to use it.

For more examples of Code Execution, refer to [this notebook](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/code-execution/intro_code_execution.ipynb).

In [35]:
code_execution_tool = Tool(code_execution=ToolCodeExecution())

response = client.models.generate_content(
    model=MODEL_ID,
    contents="Calculate 20th fibonacci number. Then find the nearest palindrome to it.",
    config=GenerateContentConfig(
        tools=[code_execution_tool],
        temperature=0,
    ),
)

display(
    Markdown(
        f"""
## Code

```py
{response.executable_code}
```

### Output

```
{response.code_execution_result}
```
"""
    )
)


## Code

```py
def fibonacci(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

fib_20 = fibonacci(20)
print(f"The 20th Fibonacci number is: {fib_20}")

```

### Output

```
The 20th Fibonacci number is: 6765

```
